# two-peak-constrained — SpectraFit Example Notebook

This user-facing notebook is the runnable companion for the `examples/two-peak-constrained` workflow.

Gaussian + pseudo-Voigt example with cross-component constraints.

## Workflow
1. Resolve local notebook paths.
2. Load `data.csv` through `sf.read(...)`.
3. Edit compact `sf.peak(...)` and `sf.background(...)` definitions.
4. Run `sf.fit(...)` and inspect the inline plot/metrics.
5. Export bundled live notebook artifacts under `outputs/live/notebook/`.


In [ ]:
from __future__ import annotations

from pathlib import Path

import spectrafit.notebook as sf


## 1 — Resolve local paths

This notebook always loads `data.csv` from the local notebook directory and writes exports under `outputs/live/notebook/`.

In [ ]:
NOTEBOOK_ROOT = Path.cwd()
DATA_PATH = NOTEBOOK_ROOT / 'data.csv'
OUTPUT_DIR = NOTEBOOK_ROOT / "outputs" / "live" / "notebook"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Notebook root: {NOTEBOOK_ROOT}")
print(f"Using local data file: {DATA_PATH}")


## 2 — Load the local dataset

Use the single notebook import to load the local spectrum and keep the data columns attached to the dataframe for the fit step.

In [ ]:
df = sf.read(DATA_PATH, x='energy', y='intensity')
location = NOTEBOOK_ROOT.name or str(NOTEBOOK_ROOT)
print(f"Loaded {DATA_PATH.name} with {len(df)} rows from {location}")
df.head()


## 3 — Define the fit and run it

The compact notebook API still compiles into the canonical `UnifiedFittingConfig -> FittingPipeline -> FitResult` chain under the hood, but the cell below hides the internal object graph.

In [ ]:
peaks = [
    sf.peak(
        'gaussian',
        id='p1',
        amplitude=(1.0, 0.0, 3.0),
        center=(-0.5, -2.0, 0.0),
        fwhmg=(0.3, 0.05, 1.0),
    ),
    sf.peak(
        'pseudovoigt',
        id='p2',
        amplitude=(0.8, 0.0, 3.0),
        center=sf.tie('p1.center + 1.0'),
        fwhmg=(0.25, 0.05, 1.0),
        fwhml=sf.tie('p2.fwhmg'),
    ),
]

background = [
    sf.background(
        'linear',
        id='bg',
        slope=sf.fixed(0.0),
        intercept=(0.02, 0.0, 0.2),
    ),
]

optimizer = sf.OptimizerConfig(
    max_nfev=1000,
    method='leastsq',
)

result = sf.fit(
    df,
    peaks=peaks,
    background=background,
    optimizer=optimizer,
    name='two-peak-constrained',
)

result.plot()
result.metrics

## 4 — Export notebook artifacts

Use one `result.save(...)` call to write the fitted dataframe, metric table, peak table, HTML plot, report, and lockfile.

In [ ]:
artifacts = result.save(OUTPUT_DIR, name='two-peak-constrained')

[path.name for path in artifacts]


## Next steps

- Inspect the generated CSVs, HTML fit plot, report, and lockfile in `outputs/live/notebook/`.
- Edit the compact `sf.peak(...)` / `sf.background(...)` definitions and rerun the fit cell to explore different models, bounds, and solver settings.
